###### 21_nlp_model_serving

###### Purpose
This notebook will use the Support ticket Serving Endpoint and test the model.

###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Serving Endpoint

- MLflow Model Serving

- REST API

- PyFunc Wrapper

###### Input
- Workspace_url

- Endpoint_url

- Token

- Headers

- Payload

######  Output
- Model prediction

######  Architecture

```text

Sample Ticket Text
      ↓
Payload
      ↓
REST API Request
      ↓
Databricks Serving Endpoint
      ↓
PyFunc Wrapper
      ↓
TF-IDF Pipeline
      ↓
Prediction Response

```

#### Section - Test the Serving endpoint

In [0]:
import requests

#Get workspace URL dynamically
workspace_url = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

print(workspace_url)

endpoint_url = (
    f"{workspace_url}/serving-endpoints/"
    "support_ticket_endpoint/invocations"
)

token = dbutils.secrets.get( scope="databricks-secrets", key="model-serving-pat")

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

payload = {
    "dataframe_records": [
        {"cleaned_ticket_text": "Internet is slow"},
        {"cleaned_ticket_text": "Password reset is not working"},
        {"cleaned_ticket_text": "My bill is too high"},
        {"cleaned_ticket_text": "I want to cancel my service"}
    ]
}

response = requests.post(endpoint_url, headers=headers, json=payload)

print(response.status_code)
print(response.text)

response = requests.post(
    endpoint_url,
    headers=headers,
    json=payload
)

print(response.json())

###### Debugging queries checklist

In [0]:
import mlflow
from mlflow.models import Model

# 1. Load the registered/logged model
loaded_model = mlflow.sklearn.load_model(model_uri)

# 2. Check model signature
model_info = Model.load(model_uri)
print(model_info.signature)

# 3. Test local predictions with list/Series
test_texts = [
    "Unable to sign in",
    "Login error on mobile app",
    "Internet is slow",
    "I want to cancel my service"
]

for text in test_texts:
    print(text, "=>", loaded_model.predict([text])[0])

# 4. Check vocabulary
tfidf = loaded_model.named_steps["tfidf"]

for word in ["unable", "sign", "login", "password", "bill", "internet", "cancel"]:
    print(word, word in tfidf.vocabulary_)


# 5. Test DataFrame vs Series behavior
import pandas as pd

test_df = pd.DataFrame({
    "cleaned_ticket_text": [
        "Internet is slow",
        "Password reset is not working",
        "My bill is too high",
        "I want to cancel my service"
    ]
})

print("DataFrame prediction:")
print(loaded_model.predict(test_df))

print("Series prediction:")
print(loaded_model.predict(test_df["cleaned_ticket_text"]))

##### Notebook Summary

- Dynamically retrieved the Databricks workspace_url

- Dynamically created endpoint_url with workspace_url and Service endpoint

- Token

- Headers

- Payload

- Sent sample support ticket text to the serving endpoint and validated predictions.

######  Key Learnings
- Mistake: Logged a raw scikit-learn TF-IDF pipeline and assumed the serving endpoint would pass the text column exactly as during training.

- Why it happened: During training, pipeline.fit() and pipeline.predict() used a pandas Series. During model serving, MLflow passed a pandas DataFrame that matched the model signature.

- Lesson Learned: Always verify that the model receives the same input type during serving as it did during training. If necessary, create a custom mlflow.pyfunc.PythonModel wrapper that extracts the required text column before calling the 
  PyFunc wrapper converts the serving DataFrame input into the Series expected by the TF-IDF pipeline.

- The sequence to be followed to debug issue:
    - Verified the model locally.
    - Verified the model locally.
    - Verified the registered model.
    - Verified the endpoint configuration.
    - Verified the model signature.
    - Compared DataFrame vs Series behavior.
    - Identified the input-type mismatch.


For setting databricks secret scope for Token, in the command prompt: 
- set DATABRICKS_HOST=https://adb-xxxxxxxxxxxxxxxx.xx.azuredatabricks.net
- set DATABRICKS_TOKEN=dapiXXXXXXXXXXXXXXXX
- databricks workspace ls /
- databricks secrets create-scope --scope databricks-secrets
- databricks secrets put --scope databricks-secrets --key model-serving-pat
- dbutils.secrets.get("databricks-secrets", "model-serving-pat")

###### Production Note

For demonstration purposes a PAT token may be used temporarily. In production, credentials should always be stored in Databricks Secret Scopes and accessed using dbutils.secrets.get().

###### Notebook Conclusion

- In this notebook, we built a real-time NLP inference service by deploying the registered Support Ticket classification model as a Databricks Serving Endpoint.

- This enables external applications to classify incoming support tickets through a REST API with low-latency predictions.

- This will be used in the next notebook to move from Traditional NLP to Modern NLP by generating embeddings for semantic understanding of support ticket text.

###### Next Notebook

22_nlp_embeddings

- The purpose of this notebook is to generate embeddings for support ticket text using an embedding model and produce dense vector representations that capture the semantic meaning of the text. These embeddings will be used in later notebooks for Vector Search, RAG, and Agentic AI applications.